# 10. LoRA & QLoRA Configuration

LoRA has a few **settings (knobs)**. Understanding them helps you get good results
without wasting GPU. Let's learn the main knobs and write a real config.

## Real-life analogy

LoRA adds small **"sticky note" layers** to the model and trains only those.
The settings decide **how big** the sticky notes are and **where** they go.

## The main knobs

| Setting | Meaning | Typical value |
|---------|---------|---------------|
| **r** (rank) | Size of the LoRA layers. Bigger = more capacity, more memory | 8, 16, 32 |
| **lora_alpha** | How strongly LoRA is applied (scaling) | often 2x `r` (16, 32) |
| **target_modules** | Which parts of the model get LoRA | attention layers like `q_proj`, `v_proj` |
| **lora_dropout** | A little randomness to avoid overfitting | 0.05 |

**QLoRA** = LoRA **plus** loading the model in **4-bit** (using `bitsandbytes`) to save memory.

## Step 1 - A LoRA config (with PEFT)

In [1]:
# !pip install peft
from peft import LoraConfig

lora_config = LoraConfig(
    r=8,                                   # rank: size of the LoRA layers
    lora_alpha=16,                         # scaling (often 2x r)
    target_modules=["q_proj", "v_proj"],   # attach LoRA to attention
    lora_dropout=0.05,                     # small dropout
    bias="none",
    task_type="CAUSAL_LM",                 # we are training a text-generation model
)
print(lora_config)

c:\Users\THIRU\OneDrive\Desktop\gen_ai_course\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


LoraConfig(task_type='CAUSAL_LM', peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, peft_version='0.19.1', base_model_name_or_path=None, revision=None, inference_mode=False, r=8, target_modules={'v_proj', 'q_proj'}, exclude_modules=None, lora_alpha=16, lora_dropout=0.05, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', trainable_token_indices=None, loftq_config={}, eva_config=None, corda_config=None, lora_ga_config=None, use_dora=False, alora_invocation_tokens=None, use_qalora=False, qalora_group_size=16, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False), lora_bias=False, target_parameters=None, use_bdlora=None, arrow_config=None, ensure_weight_tying=False)


## Step 2 - QLoRA: load the model in 4-bit (GPU only)

This part needs a **GPU** with `bitsandbytes`. It shrinks the model to 4-bit so it fits in small VRAM.
On CPU, just read it - you will use it on Colab.

In [2]:
# GPU only - run this on Colab with a GPU runtime.
from transformers import BitsAndBytesConfig
import torch

qlora_config = BitsAndBytesConfig(
    load_in_4bit=True,                      # the "Q" in QLoRA: 4-bit model
    bnb_4bit_quant_type="nf4",              # a good 4-bit format
    bnb_4bit_compute_dtype=torch.float16,   # compute in 16-bit for speed
    bnb_4bit_use_double_quant=True,         # extra memory saving
)
print(qlora_config)

# Then you would load the model like this (needs GPU):
# model = AutoModelForCausalLM.from_pretrained(model_name, quantization_config=qlora_config)

BitsAndBytesConfig {
  "_load_in_4bit": true,
  "_load_in_8bit": false,
  "bnb_4bit_compute_dtype": "float16",
  "bnb_4bit_quant_storage": "uint8",
  "bnb_4bit_quant_type": "nf4",
  "bnb_4bit_use_double_quant": true,
  "llm_int8_enable_fp32_cpu_offload": false,
  "llm_int8_has_fp16_weight": false,
  "llm_int8_skip_modules": null,
  "llm_int8_threshold": 6.0,
  "load_in_4bit": true,
  "load_in_8bit": false,
  "quant_method": "bitsandbytes"
}



## Key points to remember

- **r** (rank) sets the **size** of LoRA layers; **lora_alpha** sets their **strength**.
- **target_modules** picks **where** LoRA attaches (usually attention: `q_proj`, `v_proj`).
- **lora_dropout** (~0.05) helps avoid overfitting.
- **QLoRA** = LoRA + a **4-bit** model (`BitsAndBytesConfig`) = lowest memory, **needs a GPU**.
- Good starting point: `r=8`, `lora_alpha=16`, `lora_dropout=0.05`.